# Deliverable 6: Single-Cell RNA-Seq Analysis

This notebook prepares the environment and data for a simplified scRNA-seq pipeline following Single-cell Best Practices. It will:
- Build a cell-gene expression matrix (AnnData)
- Cluster cells with Leiden
- Annotate cell types with CellTypist

## System Requirements Check

This section checks system and Python dependencies:
- System: `cmake`, `libtbb12`, `wget`, `curl`
- Python: `scanpy`, `anndata<0.11`, `leidenalg`, `celltypist`

Notes:
- No automatic `sudo apt-get` will be executed. Missing system packages will show a suggested command for you to run manually in a terminal.
- Python packages missing will be installed with `pip --user`.


In [ ]:
import subprocess
import sys

print("\n=== Python Environment Check ===")
print(f"Python version: {sys.version}")

# Required packages (toy dataset; pin anndata <0.11)
packages = ['scanpy', 'anndata<0.11', 'leidenalg', 'celltypist', 'matplotlib']
for pkg in packages:
    display_name = pkg.split('<')[0]
    try:
        if display_name == 'anndata':
            import anndata
            print(f"[OK] anndata {anndata.__version__} installed")
        else:
            __import__(display_name)
            print(f"[OK] {display_name} installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', pkg], capture_output=True)
        try:
            __import__(display_name)
            print(f"[OK] {display_name} installed")
        except Exception:
            print(f"[WARNING] Failed to import {display_name} after installation; continue anyway.")

print("\nEnvironment check complete.")

## Data Download & Setup


In [ ]:
%%bash
set -e

DATA_DIR="data"
mkdir -p "$DATA_DIR"

echo "=== Data Setup ==="

WL_PATH="$DATA_DIR/3M-february-2018.txt"
if [ ! -f "$WL_PATH" ]; then
    echo "Downloading 10x whitelist..."
    curl -L "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz" -o "$DATA_DIR/3M-february-2018.txt.gz"
    gunzip -f "$DATA_DIR/3M-february-2018.txt.gz"
    echo "[OK] Whitelist ready"
else
    echo "[OK] Whitelist already exists"
fi

if [ -f "toy_read_ref_set.tar.gz" ]; then
    echo "Extracting sample data..."
    tar -xzf toy_read_ref_set.tar.gz -C "$DATA_DIR/"
    [ -d "$DATA_DIR/toy_read_ref_set" ] && mv "$DATA_DIR/toy_read_ref_set"/* "$DATA_DIR/" && rmdir "$DATA_DIR/toy_read_ref_set"
    echo "[OK] Data extracted"
elif [ -d "$DATA_DIR/toy_human_ref" ] && [ -d "$DATA_DIR/toy_read_fastq" ]; then
    echo "[OK] Sample data already present"
else
    echo "[ACTION REQUIRED] Please download toy_read_ref_set.tar.gz from:"
    echo "   https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814"
    echo "   Place it in the same directory as this notebook and re-run this cell."
fi

echo "Data setup complete."

## Tool Installation


In [ ]:
%%bash
set -e
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

echo "=== Tool Installation ==="

if ! command -v cargo &> /dev/null; then
    echo "Installing Rust toolchain..."
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    source "$HOME/.cargo/env"
else
    echo "[OK] Cargo already installed"
fi
export PATH="$HOME/.cargo/bin:$PATH"

if ! command -v salmon &> /dev/null; then
    echo "Installing salmon..."
    wget -q https://github.com/COMBINE-lab/salmon/releases/download/v1.10.0/salmon-1.10.0_linux_x86_64.tar.gz
    tar -xzf salmon-1.10.0_linux_x86_64.tar.gz
    mkdir -p "$HOME/.local/bin"
    mv salmon-1.10.0_linux_x86_64/bin/salmon "$HOME/.local/bin/"
    rm -rf salmon-1.10.0_linux_x86_64*
    echo "[OK] Salmon installed"
else
    echo "[OK] Salmon already installed"
fi

if ! command -v alevin-fry &> /dev/null; then
    echo "Installing alevin-fry..."
    cargo install --locked alevin-fry
    echo "[OK] Alevin-fry installed"
else
    echo "[OK] Alevin-fry already installed"
fi

if ! command -v simpleaf &> /dev/null; then
    echo "Installing simpleaf..."
    cargo install --locked simpleaf --version 0.18.3 || \
    cargo install --locked simpleaf --version 0.18.2 || \
    cargo install --locked simpleaf
    echo "[OK] Simpleaf installed"
else
    echo "[OK] Simpleaf already installed"
fi

export ALEVIN_FRY_HOME="$HOME/.alevin_fry"
mkdir -p "$ALEVIN_FRY_HOME"
if ! grep -q "ALEVIN_FRY_HOME" "$HOME/.bashrc" 2>/dev/null; then
  echo "export ALEVIN_FRY_HOME=\"$HOME/.alevin_fry\"" >> "$HOME/.bashrc"
fi

echo -e "\nTool installation complete." 
(salmon --version | head -n1) || true
(alevin-fry --version | head -n1) || true
(simpleaf --version | head -n1) || true

## Reference Indexing

In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"
export ALEVIN_FRY_HOME="$HOME/.alevin_fry"

DATA_DIR="data/toy_ref_read"
REF_DIR="$DATA_DIR/toy_human_ref"
GENOME_FA="$REF_DIR/fasta/genome.fa"
GENES_GTF="$REF_DIR/genes/genes.gtf"
INDEX_DIR="data/simpleaf_index"

printf "\n=== Reference Indexing ===\n"

# Check for required files
if [ ! -f "$GENOME_FA" ]; then
  echo "Error: Reference genome not found at $GENOME_FA"
  exit 1
fi
if [ ! -f "$GENES_GTF" ]; then
  echo "Error: GTF file not found at $GENES_GTF"
  exit 1
fi

# Skip if index already exists
if [ -d "$INDEX_DIR" ] && [ -f "$INDEX_DIR/index/complete_ref_lens.bin" ]; then
  printf "Index already exists at %s\n" "$INDEX_DIR"
  printf "Index metadata:\n"
  ls -lh "$INDEX_DIR/index/" | head -n 10
  exit 0
fi

# Set paths for simpleaf
printf "Configuring simpleaf paths...\n"
simpleaf set-paths

# Build index (read length 90 for this toy dataset)
printf "\nBuilding salmon index (this may take a few minutes)...\n"
simpleaf index \
  --output "$INDEX_DIR" \
  --fasta "$GENOME_FA" \
  --gtf "$GENES_GTF" \
  --rlen 90 \
  --threads 8 \
  --no-piscem

# Verify and show results
if [ -d "$INDEX_DIR" ] && [ -f "$INDEX_DIR/index/complete_ref_lens.bin" ]; then
  printf "\n=== Index Complete ===\n"
  printf "Index location: %s\n" "$INDEX_DIR"
  printf "Index size: %s\n" "$(du -sh "$INDEX_DIR" | cut -f1)"
  printf "\nIndex contents:\n"
  ls -lh "$INDEX_DIR/index/" | head -n 10
else
  echo "Error: Index build appears to have failed"
  exit 1
fi

## Quantification

In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"
export ALEVIN_FRY_HOME="$HOME/.alevin_fry"

DATA_DIR="data/toy_ref_read"
FASTQ_DIR="$DATA_DIR/toy_read_fastq"
INDEX_DIR="data/simpleaf_index"
QUANT_DIR="data/simpleaf_quant"
WHITELIST="data/3M-february-2018.txt"

printf "\n=== Quantification ===\n"

# Check prerequisites
if [ ! -d "$INDEX_DIR" ]; then
  echo "Error: Index not found at $INDEX_DIR. Run indexing cell first."
  exit 1
fi
if [ ! -f "$WHITELIST" ]; then
  echo "Error: Whitelist not found at $WHITELIST"
  exit 1
fi
if [ ! -d "$FASTQ_DIR" ]; then
  echo "Error: FASTQ directory not found at $FASTQ_DIR"
  exit 1
fi

# Skip if quantification already complete
if [ -d "$QUANT_DIR/af_quant/alevin" ] && [ -f "$QUANT_DIR/af_quant/alevin/quants_mat.mtx" ]; then
  printf "Quantification already complete at %s\n" "$QUANT_DIR"
  printf "\nMatrix Market header (rows x cols):\n"
  head -n 3 "$QUANT_DIR/af_quant/alevin/quants_mat.mtx"
  printf "\nCounts by file:\n"
  printf "  Barcodes (rows/cells): %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt")"
  printf "  Features (cols/genes): %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt")"
  exit 0
fi

# Collect all FASTQ files (R1 = barcode+UMI, R2 = cDNA), comma-separated and sorted
printf "Discovering FASTQ files...\n"
reads1=$(find -L "$FASTQ_DIR" -type f -name "*_R1_*.fastq*" | sort | paste -sd, -)
reads2=$(find -L "$FASTQ_DIR" -type f -name "*_R2_*.fastq*" | sort | paste -sd, -)

if [ -z "$reads1" ] || [ -z "$reads2" ]; then
  echo "Error: Could not find paired R1/R2 FASTQ files in $FASTQ_DIR"
  ls -lh "$FASTQ_DIR"
  exit 1
fi

printf "R1 (barcode) files: %s\n" "$reads1"
printf "R2 (cDNA) files:   %s\n" "$reads2"

# Run quantification in USA mode (spliced/unspliced/ambiguous) with cr-like UMI resolution
# Requires t2g_3col.tsv produced during indexing
T2G="$INDEX_DIR/index/t2g_3col.tsv"
if [ ! -f "$T2G" ]; then
  echo "Error: t2g_3col.tsv not found at $T2G (re-run indexing cell)."
  exit 1
fi

printf "\nRunning simpleaf quant (this may take several minutes)...\n"
simpleaf quant \
  -c 10xv3 -t 8 \
  -1 "$reads1" -2 "$reads2" \
  -i "$INDEX_DIR/index" \
  -u -r cr-like \
  -m "$T2G" \
  -o "$QUANT_DIR"

# Verify and show results
if [ -f "$QUANT_DIR/af_quant/alevin/quants_mat.mtx" ]; then
  printf "\n=== Quantification Complete ===\n"
  printf "Output directory: %s\n" "$QUANT_DIR/af_quant/alevin"
  printf "Total size: %s\n" "$(du -sh "$QUANT_DIR" | cut -f1)"
  
  printf "\nMatrix Market header (rows x cols):\n"
  head -n 3 "$QUANT_DIR/af_quant/alevin/quants_mat.mtx"
  
  printf "\nCounts by file (per alevin-fry spec):\n"
  printf "  Barcodes (rows/cells): %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt")"
  printf "  Features (cols/genes or gene-variants): %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt")"
  
  printf "\nSample barcodes (first 5):\n"
  head -n 5 "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt"
  
  printf "\nSample features (first 5):\n"
  head -n 5 "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt"
else
  echo "Error: Quantification appears to have failed"
  exit 1
fi

## Data Loading

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import mmread
import scipy.sparse as sp

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')

print("Loading quantification data...")
mtx_path = 'data/simpleaf_quant/af_quant/alevin/quants_mat.mtx'
rows_path = 'data/simpleaf_quant/af_quant/alevin/quants_mat_rows.txt'
cols_path = 'data/simpleaf_quant/af_quant/alevin/quants_mat_cols.txt'

X = mmread(mtx_path).tocsr()
rows = pd.read_csv(rows_path, header=None)[0].astype(str).values
cols = pd.read_csv(cols_path, header=None)[0].astype(str).values

if X.shape != (len(rows), len(cols)):
    X = X.T

adata = sc.AnnData(X)
adata.obs_names = rows
adata.var_names = cols

print(f"Matrix shape (cells x features): {adata.shape}")
print(f"Barcodes file has {len(rows)} entries (cells)")
print(f"Features file has {len(cols)} entries (genes or USA variants)")

def collapse_usa(adata):
    idx = adata.var_names
    try:
        names = idx.astype(str).values
    except Exception:
        names = np.array([str(x) for x in idx])
    bases = []
    usa_found = False
    for nm in names:
        s = str(nm)
        if '-' in s:
            base, suf = s.rsplit('-', 1)
            if suf in ('S', 'U', 'A'):
                bases.append(base)
                usa_found = True
            else:
                bases.append(s)
        else:
            bases.append(s)
    if not usa_found:
        return adata
    bases = np.array(bases, dtype=object)
    uniq, inv = np.unique(bases, return_inverse=True)
    n_feat = len(inv)
    n_gene = len(uniq)
    row_ix = np.arange(n_feat)
    col_ix = inv
    data = np.ones(n_feat, dtype=np.int8)
    M = sp.csr_matrix((data, (row_ix, col_ix)), shape=(n_feat, n_gene))
    Xc = adata.X @ M
    adata_out = sc.AnnData(Xc, obs=adata.obs.copy())
    adata_out.var_names = uniq
    return adata_out

adata = collapse_usa(adata)

# CRITICAL: Convert to float32 for proper normalization
print(f"\nConverting matrix to float32...")
print(f"Before conversion: dtype = {adata.X.dtype}")
adata.X = adata.X.astype(np.float32)
print(f"After conversion: dtype = {adata.X.dtype}")

# Verify we have raw counts
first_cell = adata.X[0].toarray().ravel() if hasattr(adata.X[0], 'toarray') else adata.X[0].ravel()
print(f"First cell raw count sum: {first_cell.sum():.2f}")

print(f"\nLoaded AnnData: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Total counts: {adata.X.sum():,.0f}")
try:
    mean_counts = float(np.asarray(adata.X.sum(axis=1)).ravel().mean()) if sp.issparse(adata.X) else float(adata.X.sum(axis=1).mean())
    print(f"  Mean counts per cell: {mean_counts:.1f}")
except Exception:
    pass

## QC and Filtering

In [ ]:
# Calculate QC metrics (toy dataset is sparse)
sc.pp.calculate_qc_metrics(adata, percent_top=[], inplace=True)

print("QC metrics calculated:")
print(f"  Mean genes per cell: {adata.obs['n_genes_by_counts'].mean():.0f}")
print(f"  Mean counts per cell: {adata.obs['total_counts'].mean():.0f}")

# More lenient thresholds for toy data
print("\nFiltering cells (min 3 genes)...")
sc.pp.filter_cells(adata, min_genes=3)
print("Filtering genes (min present in 3 cells)...")
sc.pp.filter_genes(adata, min_cells=3)

print(f"\nFiltered AnnData: {adata.n_obs} cells x {adata.n_vars} genes")

## Normalization and Preprocessing

In [ ]:
# Normalize to 10,000 counts per cell
print("Normalizing to 10,000 counts per cell...")
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
print("Log-transforming...")
sc.pp.log1p(adata)

# CRITICAL: Save log-normalized data NOW, before any other operations
adata.raw = adata.copy()
print(f"Stored log-normalized data in adata.raw: {adata.raw.shape}")

# Verify the data is correct
import numpy as np
first_cell_sum = float(np.array(adata.raw.X[0].sum()).item())
print(f"Verification - First cell sum: {first_cell_sum:.2f} (should be ~9.2)")

if first_cell_sum < 8 or first_cell_sum > 11:
    print("ERROR: Log-normalization verification failed!")
    print("Something is wrong with the normalization sequence.")
else:
    print("SUCCESS: Data is properly log-normalized")

# For this tiny toy dataset, mark all genes as highly variable
adata.var['highly_variable'] = True
print(f"Marked all {adata.n_vars} genes as highly variable")

# Subset to HVGs (which is all genes for toy data)
adata = adata[:, adata.var['highly_variable']]

print(f"\nAfter HVG filtering: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"adata.raw preserved: {adata.raw.n_obs} cells x {adata.raw.n_vars} genes")

## Dimensionality Reduction and Clustering

In [ ]:
# Scale data
print("Scaling data...")
sc.pp.scale(adata, max_value=10)

# PCA with adaptive number of components
n_comps = int(min(40, max(2, adata.n_vars - 1)))
print(f"Computing PCA ({n_comps} components)...")
sc.tl.pca(adata, n_comps=n_comps, random_state=0)

# Compute neighbors graph with adaptive PCs and k
k = int(min(10, max(2, adata.n_obs - 1)))
print(f"Computing neighbors graph (k={k}, n_pcs={n_comps})...")
sc.pp.neighbors(adata, n_neighbors=k, n_pcs=n_comps, random_state=0)

# UMAP embedding
print("Computing UMAP embedding...")
sc.tl.umap(adata, random_state=0)

# Leiden clustering via igraph backend (future Scanpy default)
print("Running Leiden clustering (igraph backend, resolution=0.5)...")
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False, random_state=0)

print(f"\nClustering complete: {adata.obs['leiden'].nunique()} clusters identified")

## Cell Type Annotation

In [ ]:
import celltypist
from celltypist import models

# Download and load model
models.download_models(model='Immune_All_Low.pkl', force_update=False)
model = models.Model.load(model='Immune_All_Low.pkl')

# Always use Ensembl IDs (no gene symbol mapping)
adata_for_ct = adata
print(f"Using Ensembl IDs: {adata_for_ct.n_obs} cells x {adata_for_ct.raw.n_vars} genes")

# Final verification before CellTypist
import numpy as np
if hasattr(adata_for_ct, 'raw') and adata_for_ct.raw is not None:
    check_data = adata_for_ct.raw.X
else:
    check_data = adata_for_ct.X

first_sum = float(np.array(check_data[0].sum()).item())
print(f"Data check - First cell sum: {first_sum:.2f}")

if first_sum < 8 or first_sum > 11:
    print("ERROR: Data is not log-normalized. CellTypist will fail.")
    print("Check your normalization section.")
    adata.obs['cell_type'] = 'Unknown'
    adata.obs['cell_type_raw'] = 'Unknown'
else:
    print("Data looks good. Running CellTypist...")
    try:
        predictions = celltypist.annotate(adata_for_ct, model='Immune_All_Low.pkl', majority_voting=True)
        adata.obs['cell_type'] = predictions.predicted_labels.majority_voting
        adata.obs['cell_type_raw'] = predictions.predicted_labels.predicted_labels
        print(f"\nCell type annotation complete")
        print(f"Identified {adata.obs['cell_type'].nunique()} unique cell types:")
        for ct in sorted(adata.obs['cell_type'].unique()):
            count = (adata.obs['cell_type'] == ct).sum()
            print(f"  {ct}: {count} cells")
    except Exception as e:
        print(f"\nCellTypist failed: {e}")
        adata.obs['cell_type'] = 'Unknown'
        adata.obs['cell_type_raw'] = 'Unknown'

## Visualization

In [ ]:
import matplotlib.pyplot as plt

# Create side-by-side visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot Leiden clustering
sc.pl.umap(adata, color=['leiden'], legend_loc='on data', 
           title='Leiden Clustering', ax=ax1, show=False)

# Plot cell type annotation
sc.pl.umap(adata, color=['cell_type'], 
           title='Cell Type Annotation', ax=ax2, show=False)

plt.tight_layout()
plt.show()

print(f"\nClusters: {', '.join(sorted(adata.obs['leiden'].unique()))}")

In [ ]:
# Individual cell type annotation plot
sc.pl.umap(adata, color='cell_type', title='Cell Type Annotation', show=True)

# Verification checks
print("\n=== Final Verification ===")
print(f"1. adata.raw exists: {adata.raw is not None}")
if adata.raw is not None:
    print(f"   adata.raw shape: {adata.raw.shape}")
    first_sum = float(np.array(adata.raw.X[0].sum()).item())
    print(f"   First cell sum in raw: {first_sum:.2f} (should be ~9.2)")

print(f"\n2. Leiden clusters found: {adata.obs['leiden'].nunique()}")
print(f"   Cluster labels: {sorted(adata.obs['leiden'].unique())}")

print(f"\n3. Cell types found: {adata.obs['cell_type'].nunique()}")
print(f"   Cell type labels: {sorted(adata.obs['cell_type'].unique())}")

if adata.obs['cell_type'].nunique() == 1 and 'Unknown' in adata.obs['cell_type'].values:
    print("\n   WARNING: All cells labeled as 'Unknown' - CellTypist did not work correctly")
    print("   Check that adata.raw.X contains log-normalized data")
else:
    print("\n   SUCCESS: CellTypist identified distinct cell types")

# Final Summary
print("\n" + "=" * 60)
print("SINGLE-CELL RNA-SEQ ANALYSIS COMPLETE")
print("=" * 60)
print(f"\nDataset shape: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"Number of Leiden clusters: {len(adata.obs['leiden'].unique())}")
print(f"Number of unique cell types identified: {len(adata.obs['cell_type'].unique())}")
print(f"\nCell type distribution:")
print(adata.obs['cell_type'].value_counts())
print("\n" + "=" * 60)